[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/mhd_sphere/mhd_sphere.ipynb)

# Magnetohydrodynamics on a Sphere

This example computes magnetohydrodynamics on a sphere: a thin layer of conducting fluid, like the outer layer of a star, whose flow carries the magnetic field threaded through it while the field pulls back on the flow. A conducting fluid drags its field lines along as if they were frozen into it, so a shearing flow winds the lines up and strengthens the field; the field resists being bent, as a stretched string does, and turns the flow in return. Where oppositely directed lines are pressed together, the current gathers in thin sheets and the field breaks and reconnects into closed loops, magnetic islands. Here a differential rotation, the equator turning against the middle latitudes as on the sun, winds up a field whose lines run from pole to pole.

The sphere lives in spacetime, `x+y+z+t-`, in one instant: its points are directions of space, and a fluid parcel's worldline is time joined with its velocity. On a surface an incompressible flow and a field without sources are each one function on the vertices, the stream function and the flux function, whose contours are streamlines and field lines. The flux function's spacetime gradient is the field: paired with an edge, `gradient(flux) & edge`, it is the magnetic flux through the wall standing on that edge, and paired with time it is the electric field along the surface's normal. A perfect conductor freezes the field into the fluid, which says that the gradient pairs to nothing with every worldline; resistivity is what lets that pairing slip. The flow carries its vorticity, and the field's tension turns it, through one bracket, the Jacobian of two functions on each triangle: the gradient of one paired with the velocity of the other. The bracket is antisymmetric, so without viscosity or resistivity the total energy, the cross helicity and the mean square flux are kept exactly.

In the vorticity–flux form of two-dimensional magnetohydrodynamics this reads as $\partial_t \omega + [\phi, \omega] = [\psi, j] + \nu \nabla^2 \omega$ and $\partial_t \psi + [\phi, \psi] = \eta \nabla^2 \psi$, with the vorticity $\omega = -\nabla^2 \phi$ of the stream function $\phi$, the current $j = -\nabla^2 \psi$ of the flux function $\psi$, and $[a, b]$ the Jacobian of two functions on the surface; the second equation is the frozen-in condition $(\partial_t + \mathbf u \cdot \nabla)\psi = 0$, slipped by resistivity.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import islice, tee

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import save_figure, show_animation
from examples.fluids.mhd_sphere import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# The planes of space, which read a point's coordinates; a function's gradient on a triangle is one.
Plane = ga.gatype.from_blades("x y z")              # Vector+
# The points of space, each the direction from the origin to it; the sphere's points, its edges and the fluid's
# velocities are points too.
Point = Plane.dual().output_gatype                  # Antivector-
# A function integrated over each vertex's share of the surface is an amount.
Amount = ga.gatype.antiscalar()                     # Antiscalar-

## The sphere in spacetime

A triangle's edges are directions of space, and their join is its patch. A corner's hat function rises from nothing on the edge facing it to one at the corner, and its gradient is that edge over the patch. Met with the triangle's unit patch, a stream function's gradient gives the velocity along its contours, and the stiffness is the flow's kinetic energy, its momentum paired with its velocity, `momentum & velocity`, pulled back to the vertices: the negated Laplacian, against each vertex's share of the surface. A stream function's constant is free, and pinning one vertex fixes it.

In [ ]:
# The sphere's subdivision levels.
n_levels = 5

In [ ]:
topology, coordinates = shapes.sphere(n_levels)
vertices = mv(Plane, coordinates).dual().field()                               # Point-[vertices]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                     # [corners] Point-[faces]
patches = (sides[0] & sides[1]) / 2                                              # Patch-[faces]
areas = patches.squared().square_root()                                          # Scalar+[faces]
corner_gradients = (sides | patches.inverse()) / 2                               # [corners] Plane+[faces]
gradient = (spdiag(corner_gradients * Scalar) * topology.incidence[2, 0]).sum(axis=0)   # (Plane+ <- Scalar+)[faces <- vertices]
velocity = spdiag(Plane ^ (patches / areas))(gradient)                         # (Point- <- Scalar+)[faces <- vertices]

# Each corner's share of the faces around it, and the map handing each vertex its share of a face integral.
corner_mean = topology.incidence[2, 0].mean(axis=0)                              # Scalar+[faces <- vertices]
dual_areas = ~corner_mean * areas                                                # Scalar+[vertices]
lumped = spdiag(1 / dual_areas) * ~corner_mean                                   # Scalar+[vertices <- faces]
momentum = spdiag(areas * Point.dual())                                      # (Plane+ <- Point-)[faces <- faces]
stiffness = velocity.adjugate()(momentum(velocity))                              # (Amount- <- Scalar+)[vertices <- vertices]
mass = spdiag(dual_areas * Scalar.dual())                                        # (Amount- <- Scalar+)[vertices <- vertices]
reference = mv.scalar((np.arange(topology.n_vertices) == 0)[:, None]).field()    # Scalar+[vertices]
pinned = stiffness + spdiag(stiffness.diagonal() * reference)                    # (Amount- <- Scalar+)[vertices <- vertices]

## The bracket and the frozen field

A flow carries a function along at the rate of its gradient paired with the flow's velocity, taken away as the flow brings in what lies upstream; on each triangle the stream function's gradient met with the area patch is its area-weighted velocity, and the pairing is the Jacobian integrated over the triangle. The vorticity is carried by its own flow and turned by the field, the bracket with the flux function standing where the stream function stood: field lines bent into a curve pull straight as stretched strings do. The flux function obeys the frozen-in condition itself, its spacetime gradient pairing to nothing with each worldline, solved for its rate along time; it is the same bracket, written as the spacetime statement it is.

In [ ]:
def bracket(stream: Scalar, carried: Scalar) -> Scalar:
    """The rate at which the flow of a stream function carries a function, at the vertices."""
    flows = gradient(stream) ^ patches                                           # Point-[faces]
    return -(lumped * (gradient(carried) & flows))                               # Scalar+[vertices]


def laplacian(function: Scalar) -> Scalar:
    """The negated Laplacian at the vertices: the stiffness's amount over each vertex's share of the surface."""
    return mass.solve(stiffness(function))                                       # Scalar+[vertices]


def rate(state: Scalar, viscosity: float, resistivity: float) -> Scalar:
    """The rate of change of the vorticity and flux functions `[2] Scalar+[vertices]`."""
    vorticity, flux = state                                                      # Scalar+[vertices], Scalar+[vertices]
    stream, current = pinned.solve(mass(vorticity)), laplacian(flux)             # Scalar+[vertices], Scalar+[vertices]
    vorticity_rate = bracket(stream, vorticity) - bracket(flux, current) - viscosity * laplacian(vorticity)   # Scalar+[vertices]
    # Frozen in: the flux function's spacetime gradient pairs to nothing with each worldline, time joined with the
    # velocity, so its rate along time is minus its change along the velocity, over each triangle.
    flows = gradient(stream) ^ patches                                           # Point-[faces]
    frozen = -(gradient(flux) & flows)                                           # Scalar+[faces]
    flux_rate = lumped * frozen - resistivity * current                          # Scalar+[vertices]
    return stack((vorticity_rate, flux_rate))                                    # [2] Scalar+[vertices]


def evolve(initial: Scalar, viscosity: float, resistivity: float, dt: float, n_steps: int) -> Iterator[Scalar]:
    """Successive vorticity and flux functions `[2] Scalar+[vertices]` follow classical fourth-order Runge–Kutta
    steps."""
    state = initial                                                              # [2] Scalar+[vertices]
    yield state
    for _ in range(n_steps):
        first = rate(state, viscosity, resistivity)                              # [2] Scalar+[vertices]
        second = rate(state + (dt / 2) * first, viscosity, resistivity)          # [2] Scalar+[vertices]
        third = rate(state + (dt / 2) * second, viscosity, resistivity)          # [2] Scalar+[vertices]
        fourth = rate(state + dt * third, viscosity, resistivity)                # [2] Scalar+[vertices]
        state = state + (dt / 6) * (first + 2 * second + 2 * third + fourth)     # [2] Scalar+[vertices]
        yield state

## Differential rotation winding up the field

The flow starts as differential rotation, a zonal stream function of degree three: the equator turns one way and the middle latitudes the other. The field starts with its lines running from pole to pole, the contours of the flux function `mv.x & vertices`, which circle the two points where the x axis meets the equator. The flow's energy is fifty times the field's, so at first the field is carried along almost passively. This cell takes about half a minute, most of it the Poisson solves for the stream function, four in every step; its frames show as they are computed.

In [ ]:
# The differential rotation's strength, and the strength of the field it winds up.
rotation = 1.0
field_strength = 0.25
# Viscosity and resistivity, the Runge–Kutta step, the steps run and the stride between frames, and the frame duration.
viscosity = 1e-3
resistivity = 1e-3
dt = 0.02
n_steps = 450
stride = 4
frame_duration_ms = 40

In [ ]:
heights = mv.z & vertices                                                        # Scalar+[vertices]
stream = rotation * (5 * heights * heights * heights - 3 * heights) / 2          # Scalar+[vertices]
flux = field_strength * (mv.x & vertices)                                        # Scalar+[vertices]
start = stack((laplacian(stream), flux))                                         # [2] Scalar+[vertices]
# Every stride-th state, drawn as it comes and kept for the energies.
drawn, kept = tee(islice(evolve(start, viscosity, resistivity, dt, n_steps), 0, None, stride))


def frames(states: Iterable[Scalar]) -> Iterator[tuple[Scalar, Scalar, Scalar]]:
    """The current and flux function of each state, and the field's strength on each face: the steepness of the flux
    function, how much flux crosses a wall of unit length."""
    for vorticity, flux in states:
        yield laplacian(flux), flux, gradient(flux).norm()


show_animation(render.animate(topology, vertices, frames(drawn)), "mhd_sphere", frame_duration_ms,
               n_frames=n_steps // stride + 1)

The sphere is coloured by the current, orange where it flows one way and violet the other, and the dark lines are field lines. The rotation shears the lines into bands wrapped around the sphere, and the current gathers in thin sheets between oppositely directed lines; there the lines break and reconnect into magnetic islands, which merge as the field relaxes.

## Energy traded between flow and field

Winding up the field stretches its lines, and the work of stretching them is energy taken from the flow; the field's tension hands it back as the lines unwind, while viscosity and resistivity drain both. The kinetic and magnetic energies are half the stiffness's amounts of the stream and flux functions, each paired with its function: half their squared gradients, integrated over the sphere.

In [ ]:
states = stack(tuple(kept))                                                      # [frames, 2] Scalar+[vertices]
vorticities, fluxes = states[:, 0], states[:, 1]                                 # [frames] Scalar+[vertices], [frames] Scalar+[vertices]
streams = pinned.solve(mass(vorticities))                                        # [frames] Scalar+[vertices]
kinetic = (streams * stiffness(streams)).sites.sum() / 2                         # [frames] Amount-
magnetic = (fluxes * stiffness(fluxes)).sites.sum() / 2                          # [frames] Amount-
save_figure(render.draw_energies(np.arange(0, n_steps + 1, stride) * dt, kinetic, magnetic), "mhd_sphere_energies");

In [ ]:
# checks
# Without viscosity and resistivity the flow and field trade energy and keep the total, the cross helicity and the
# mean square flux: at a state from the run, the rates pair to nothing with each.
vorticity, flux = states[len(states) // 4]                                       # Scalar+[vertices], Scalar+[vertices]
vorticity_rate, flux_rate = rate(stack((vorticity, flux)), 0.0, 0.0)             # Scalar+[vertices], Scalar+[vertices]
stream, current = pinned.solve(mass(vorticity)), laplacian(flux)                 # Scalar+[vertices], Scalar+[vertices]
unit = mv.scalar([1.0])
energy_rate = (stream * mass(vorticity_rate) + current * mass(flux_rate)).sites.sum() & unit
cross_helicity_rate = (flux * mass(vorticity_rate) + vorticity * mass(flux_rate)).sites.sum() & unit
flux_square_rate = (flux * mass(flux_rate)).sites.sum() & unit
# The scale of each rate: the energy the flow carries through the field, as fast as it does.
scale = ((current * mass(bracket(stream, flux))).sites.sum() & unit).abs()
for invariant_rate in (energy_rate, cross_helicity_rate, flux_square_rate):
    np.testing.assert_allclose(invariant_rate.to_array(), 0.0, atol=1e-13 * scale.to_array())
# With them, the total energy only falls.
assert np.all(np.diff(((kinetic + magnetic) & unit).to_array()) < 0)